In [1]:
import torch
from pathlib import Path
from torchvision.transforms import v2 as transforms_v2

In [ ]:
from datetime import datetime

TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [2]:
RAW_DATA_DIR: Path = Path("data/cub")
LOG_DIR: Path = Path("logs")

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE

In [ ]:
!nvidia-smi

# Setup dataloaders

In [5]:
import torch
from torch.utils.data import DataLoader

In [6]:
BATCH_SIZE: int = 256
NUM_WORKERS: int = 8

IMAGENET1K_MEAN = [0.485, 0.456, 0.406]
IMAGENET1K_STD = [0.229, 0.224, 0.225]

In [7]:
# V4 Pip-Net
train_transforms = transforms_v2.Compose(
    [
        transforms_v2.Resize(size=(224 + 8, 224 + 8), antialias=True),
        transforms_v2.RandomHorizontalFlip(),
        transforms_v2.RandomCrop(size=(224, 224)),
        transforms_v2.TrivialAugmentWide(),
        transforms_v2.ToImage(),
        transforms_v2.RandomErasing(p=0.1),
        transforms_v2.ToDtype(torch.float32, scale=True),
        transforms_v2.Normalize(mean=IMAGENET1K_MEAN, std=IMAGENET1K_STD),
    ]
)


test_transforms = transforms_v2.Compose(
    [
        transforms_v2.Resize(size=(224 + 8, 224 + 8), antialias=True),
        transforms_v2.CenterCrop(224),
        transforms_v2.ToImage(),
        transforms_v2.ToDtype(torch.float32, scale=True),
        transforms_v2.Normalize(mean=IMAGENET1K_MEAN, std=IMAGENET1K_STD),
    ]
)

In [8]:
import os
import pandas as pd
from torchvision.datasets import VisionDataset
from torchvision.io import read_image


class CUB200(VisionDataset):
    def __init__(self, root, train=True, transform=None, target_transform=None):
        super().__init__(root, transform=transform, target_transform=target_transform)

        self.root = os.path.join(root, "CUB_200_2011")
        self.train = train

        # Load metadata
        image_paths = pd.read_csv(
            os.path.join(self.root, "images.txt"), sep=" ", names=["img_id", "path"]
        )
        labels = pd.read_csv(
            os.path.join(self.root, "image_class_labels.txt"),
            sep=" ",
            names=["img_id", "label"],
        )
        train_test_split = pd.read_csv(
            os.path.join(self.root, "train_test_split.txt"),
            sep=" ",
            names=["img_id", "is_train"],
        )

        # Merge metadata
        data = image_paths.merge(labels, on="img_id").merge(
            train_test_split, on="img_id"
        )

        # Select train or test split
        self.data = data[data["is_train"] == int(train)]

        # Convert labels to zero-based index
        self.data["label"] -= 1

        self.image_paths = self.data["path"].tolist()
        self.labels = self.data["label"].tolist()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        img_path = os.path.join(self.root, "images", self.image_paths[index])
        label = self.labels[index]

        # Use torchvision.io for faster image loading
        image = read_image(img_path).float() / 255.0  # Normalize to [0,1]

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            label = self.target_transform(label)

        return image, label

In [ ]:
train_dataset = CUB200(
    root=RAW_DATA_DIR,
    train=True,
    transform=train_transforms,
)

test_dataset = CUB200(
    root=RAW_DATA_DIR,
    train=False,
    transform=test_transforms,
)

In [10]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

# Setup model

In [11]:
import torch
import torch.nn as nn
from torchvision import models

In [12]:
model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
_ = model.to(DEVICE)

In [13]:
NUM_CLASSES: int = 200
EMBED_DIM: int = 768

In [14]:
model.classifier.pop(-1)
model.classifier.append(nn.Linear(EMBED_DIM, NUM_CLASSES))
_ = model.to(DEVICE)

# Setup loss & optimizer

In [15]:
LR: float = 1e-4
WD: float = 1e-4
LABEL_SMOOTHING = 0.1

In [16]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

# Train

In [17]:
from tqdm import tqdm

In [18]:
LOG_FILEPATH = LOG_DIR / "training_log.csv"
train_log = pd.DataFrame(
    columns=["epoch", "phase", "train_loss", "train_acc", "test_acc"]
)

In [19]:
from torchvision.transforms import v2

cutmix = v2.CutMix(num_classes=NUM_CLASSES)
mixup = v2.MixUp(num_classes=NUM_CLASSES)
cutmix_or_mixup = v2.RandomChoice([cutmix, mixup])


@torch.no_grad()
def test_epoch(
    model: nn.Module,
    data_loader: DataLoader,
):
    model.eval()
    correct, total = 0, 0

    for x, y in tqdm(data_loader, total=len(data_loader), desc="Test"):
        x, y = x.to(DEVICE), y.to(DEVICE)

        logits = model(x)
        _, y_pred = torch.max(logits, 1)
        correct += (y_pred == y).sum().cpu().item()
        total += y.size(0)

    return correct / total


def train_epoch(
    model: nn.Module,
    data_loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
):
    model.train()

    running_loss = 0.0
    correct, total = 0, 0

    for x, y in tqdm(data_loader, total=len(data_loader), desc="Train"):
        x, y = x.to(DEVICE), y.to(DEVICE)

        x, y = cutmix_or_mixup(x, y)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            running_loss += loss.item()
            _, y_pred = torch.max(logits, 1)
            _, y = torch.max(y, 1)
            correct += (y_pred == y).sum().item()
            total += y.size(0)

    train_acc = correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_acc

In [20]:
def update_optimizer_params(
    phase: str,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
):
    trainable_params = []

    if phase == "linear_probe":
        trainable_params = list(model.classifier.parameters())

    elif phase == "finetune":
        trainable_params = list(model.classifier.parameters()) + list(
            model.features[-1].parameters()
        )

    elif phase == "full":
        trainable_params = list(model.parameters())
    else:
        raise NotImplementedError(f"phase {phase} not implemented!")

    for param in model.parameters():
        param.requires_grad = False

    for param in trainable_params:
        param.requires_grad = True

    optimizer.param_groups[0]["params"] = trainable_params

In [21]:
from typing import Callable


def train_phase(
    phase_name: str,
    num_epochs: int,
    model: nn.Module,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    test_loader: DataLoader,
    train_log: pd.DataFrame,
    early_stopper: Callable,
):
    update_optimizer_params(
        phase=phase_name,
        model=model,
        optimizer=optimizer,
    )

    for epoch in range(num_epochs):
        print(f"Phase: {phase_name} | Epoch: {epoch}")

        train_loss, train_acc = train_epoch(
            model=model,
            data_loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
        )
        print(
            f"Train: Acc {str(round(train_acc, 3))} | Loss {str(round(train_loss, 3))}"
        )

        test_acc = test_epoch(
            model=model,
            data_loader=test_loader,
        )
        print(f"Test: Acc {str(round(test_acc, 3))}")

        # Log results
        curr_log = pd.DataFrame(
            {
                "epoch": [epoch],
                "phase": [phase_name],
                "train_loss": [train_loss],
                "train_acc": [train_acc],
                "test_acc": [test_acc],
            },
        )

        train_log = pd.concat([train_log, curr_log])
        train_log.to_csv(LOG_FILEPATH, index=False)

        if early_stopper(test_acc):
            print("Early stopping")
            break
        else:
            torch.save(model.state_dict(), f"temp_checkpoint_{TIMESTAMP}.pth")

In [ ]:
class EarlyStopping:
    def __init__(self, patience: int):
        self.patience = patience
        self.counter = 0
        self.best_accuracy = None
        self.early_stop = False

    def __call__(self, accuracy: float) -> bool:
        if self.best_accuracy is None:
            self.best_accuracy = accuracy
        elif accuracy < self.best_accuracy:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_accuracy = accuracy
            self.counter = 0

        return self.early_stop

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2e-5)
train_phase(
    phase_name="linear_probe",
    num_epochs=999,
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=train_loader,
    test_loader=test_loader,
    train_log=train_log,
    early_stopper=EarlyStopping(patience=5),
)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=2e-5)

# load the best checkpoint from the previous phase
model.load_state_dict(torch.load(f"temp_checkpoint_{TIMESTAMP}.pth"))

train_phase(
    phase_name="full",
    num_epochs=999,
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    train_loader=train_loader,
    test_loader=test_loader,
    train_log=train_log,
    early_stopper=EarlyStopping(patience=10),
)